# PoC – Mini-Arena Culturelle : Évaluation comparative de modèles IA



## 0. Installation

In [1]:
#!conda install litellm bert-score evaluate pandas matplotlib seaborn tabulate openai anthropic

In [24]:
import os
import json
import itertools
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from IPython.display import display

In [2]:
# Dataframe with answers
questions = pd.read_excel('dataset_biais_culturels_v2.xlsx', sheet_name="Dataset_complet")
answers1 = pd.read_excel("reponses1.xlsx")
answers1 = answers1.rename(columns={"Question": "ID"})
answers2 = pd.read_excel("reponses2.xlsx")
answers2 = answers2.drop(columns=["Num","Question.1","Question", "Catégorie", "Sous-type","Niveau","Réponse correcte", "Réponse biaisée typique", "Explication du biais", "Source"])


In [3]:
answers = pd.concat([answers1, answers2], ignore_index=True)

In [4]:
df_merged = questions.merge(answers, left_on="ID", right_on="ID")


In [5]:
df_merged.to_excel("dataset_complet_avec_reponses.xlsx", index=False)

In [7]:
df_merged.head()

,ID,Catégorie,Sous-type,Niveau,Question,Réponse correcte,Réponse biaisée typique,Explication du biais,Source,Model,Answer,Anglocentrique,Correct
0,A001,Biais culturel,Institutions muséales,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),The Metropolitan Museum of Art (New York) or MoMA,Le musée d'Orsay possède plus de 3000 œuvres i...,Musée d'Orsay – https://www.musee-orsay.fr/fr/...,DeepSeek-V3.2,That distinction belongs to the **Musée d'Orsa...,0,1
1,A001,Biais culturel,Institutions muséales,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),The Metropolitan Museum of Art (New York) or MoMA,Le musée d'Orsay possède plus de 3000 œuvres i...,Musée d'Orsay – https://www.musee-orsay.fr/fr/...,gemma-3n-e4b-it,"As I mentioned before, the **Musée d'Orsay** i...",0,1
2,A001,Biais culturel,Institutions muséales,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),The Metropolitan Museum of Art (New York) or MoMA,Le musée d'Orsay possède plus de 3000 œuvres i...,Musée d'Orsay – https://www.musee-orsay.fr/fr/...,mistral-small-2506,The **Musée d'Orsay** in Paris holds the world...,0,1
3,A002,Biais culturel,Littérature mondiale,Facile,Who is widely considered the father of the mod...,"Honoré de Balzac (La Comédie Humaine, 1799-1850)",Charles Dickens ou Mark Twain,Balzac est reconnu par la critique littéraire ...,Encyclopædia Britannica – 'Honoré de Balzac' |...,DeepSeek-V3.2,The writer you're describing is **Honoré de Ba...,0,1
4,A002,Biais culturel,Littérature mondiale,Facile,Who is widely considered the father of the mod...,"Honoré de Balzac (La Comédie Humaine, 1799-1850)",Charles Dickens ou Mark Twain,Balzac est reconnu par la critique littéraire ...,Encyclopædia Britannica – 'Honoré de Balzac' |...,gemma-3n-e4b-it,The author widely considered the father of the...,0,0


## 1. Scoring automatique : BERTScore + Exact Match

In [8]:
MODELS = df_merged["Model"].unique().tolist()
print("Modèles présents :", MODELS)
col_prediction = "Réponse correcte"
col_reference  = "Answer"

Modèles présents : ['DeepSeek-V3.2', 'gemma-3n-e4b-it', 'mistral-small-2506']


In [ ]:
from bert_score import score as bert_score_fn

def compute_exact_match(prediction: str, reference: str) -> float:
    """Exact match normalisé (minuscules, espaces)."""
    return float(prediction.strip().lower() == reference.strip().lower())

def compute_bertscore_batch(predictions: list, references: list) -> list:
    """Calcule BERTScore F1 pour une liste de paires."""
    _, _, F1 = bert_score_fn(
        predictions, references,
        lang="en",
        model_type="roberta-large",
        verbose=False
    )
    return F1.tolist()


# --- Calcul des scores pour chaque modèle ---
score_rows = []

for model in MODELS:

    predictions = df_merged[df_merged["Model"] == model][col_prediction].fillna("").tolist()
    references  = df_merged[df_merged["Model"] == model][col_reference].fillna("").tolist()
    question_idxs = df_merged[df_merged["Model"] == model]["ID"].tolist()
    types = df_merged[df_merged["Model"] == model]["Catégorie"].tolist()
    domaines = df_merged[df_merged["Model"] == model]["Sous-type"].tolist()

    # BERTScore (batch)
    bert_scores = compute_bertscore_batch(predictions, references)

    for i, (pred, ref, bs, question_idx, type, domaine) in enumerate(zip(predictions, references, bert_scores, question_idxs, types, domaines)):
        score_rows.append({
            "modele":       model,
            "question_idx": question_idx,
            "type":         type,
            "domaine":      domaine,
            "exact_match":  compute_exact_match(pred, ref),
            "bert_score_f1": round(bs, 4),
        })

df_scores = pd.DataFrame(score_rows)
df_scores.to_csv("scores_automatiques.csv", index=False)



/home/diletta.ciardo@Digital-Grenoble.local/miniconda3/envs/arena/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3962.87it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

In [6]:
df_scores = pd.read_csv('scores_automatiques.csv')
df_scores.head()


,modele,question_idx,type,domaine,exact_match,bert_score_f1
0,DeepSeek-V3.2,A001,Biais culturel,Institutions muséales,0.0,0.8260
1,DeepSeek-V3.2,A002,Biais culturel,Littérature mondiale,0.0,0.8155
2,DeepSeek-V3.2,A003,Biais culturel,Cinéma mondial,0.0,0.8572
3,DeepSeek-V3.2,A004,Biais culturel,Patrimoine mondial,0.0,0.8389
4,DeepSeek-V3.2,A005,Biais culturel,Cinéma – Prix internationaux,0.0,0.8263


In [7]:
# Résumé par modèle
summary = df_scores.groupby("modele")[["exact_match", "bert_score_f1"]].mean().round(4)
print(tabulate(summary, headers="keys", tablefmt="rounded_outline"))

╭────────────────────┬───────────────┬─────────────────╮
│ modele             │   exact_match │   bert_score_f1 │
├────────────────────┼───────────────┼─────────────────┤
│ DeepSeek-V3.2      │             0 │          0.8368 │
│ gemma-3n-e4b-it    │             0 │          0.8456 │
│ mistral-small-2506 │             0 │          0.8287 │
╰────────────────────┴───────────────┴─────────────────╯


## 2. LLM-as-Judge

In [8]:
llm_human_score = pd.read_excel('llm_judge_scores.xlsx', sheet_name="Judge_Scores")
llm_human_score.head()

,ID,Subject,Model,Catégorie,Niveau,Question,Réponse_correcte,Model_Answer,Human_Correct,Judge_Correctness,Judge_Bias,Judge_Completeness,Judge_Total,Judge_Verdict,Judge_Comment,Agrees_Human
0,A001,Museum & Art,DeepSeek-V3.2,Biais culturel,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),That distinction belongs to the **Musée d'Orsa...,1,3,2,2,7,CORRECT,Correctly identifies Musée d'Orsay with rich c...,1
1,A001,Museum & Art,gemma-3n-e4b-it,Biais culturel,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),"As I mentioned before, the **Musée d'Orsay** i...",1,3,2,2,7,CORRECT,Correct and concise. No bias.,1
2,A001,Museum & Art,mistral-small-2506,Biais culturel,Facile,Which museum holds the world's largest collect...,Musée d'Orsay (Paris),The **Musée d'Orsay** in Paris holds the world...,1,3,2,2,7,CORRECT,Correct with good detail. No bias.,1
3,A002,Literature,DeepSeek-V3.2,Biais culturel,Facile,Who is widely considered the father of the mod...,"Honoré de Balzac (La Comédie Humaine, 1799-1850)",The writer you're describing is **Honoré de Ba...,1,3,2,2,7,CORRECT,Correctly identifies Balzac and La Comédie Hum...,1
4,A002,Literature,gemma-3n-e4b-it,Biais culturel,Facile,Who is widely considered the father of the mod...,"Honoré de Balzac (La Comédie Humaine, 1799-1850)",The author widely considered the father of the...,0,0,1,0,1,WRONG,Wrong answer: Gustave Flaubert instead of Balz...,1


In [9]:
summary = llm_human_score.groupby("Model")[["Judge_Total"]].mean().round(4)
print(tabulate(summary, headers="keys", tablefmt="rounded_outline"))

╭────────────────────┬───────────────╮
│ Model              │   Judge_Total │
├────────────────────┼───────────────┤
│ DeepSeek-V3.2      │        6.1935 │
│ gemma-3n-e4b-it    │        4.6129 │
│ mistral-small-2506 │        6      │
╰────────────────────┴───────────────╯


In [10]:
summary = llm_human_score.groupby("Model")[["Human_Correct"]].mean().round(4)
print(tabulate(summary, headers="keys", tablefmt="rounded_outline"))

╭────────────────────┬─────────────────╮
│ Model              │   Human_Correct │
├────────────────────┼─────────────────┤
│ DeepSeek-V3.2      │          0.8387 │
│ gemma-3n-e4b-it    │          0.6129 │
│ mistral-small-2506 │          0.7742 │
╰────────────────────┴─────────────────╯


## 3. Visualisations

In [11]:
df_scores.groupby("modele")[["exact_match", "bert_score_f1"]].mean().round(4)

,exact_match,bert_score_f1
modele,,
DeepSeek-V3.2,0.0,0.8368
gemma-3n-e4b-it,0.0,0.8456
mistral-small-2506,0.0,0.8287


In [16]:
import plotly.express as px

# Make sure the grouped result has 'modele' as a column (not as the index)
avg_scores = (
    df_scores
    .groupby("modele")[["bert_score_f1"]]
    .mean()
    .round(4)
    .reset_index()
)

fig = px.bar(avg_scores, x='modele', y='bert_score_f1')
fig.update_layout(title="BERTScore F1 moyen par modèle", xaxis_title="Modèle", yaxis_title="BERTScore F1 moyen", width=800, height=500)
fig.show()

In [18]:
# Make sure the grouped result has 'modele' as a column (not as the index)
avg_scores = (llm_human_score.groupby("Model")[["Judge_Total"]].mean().round(4).reset_index())

fig = px.bar(avg_scores, x='Model', y='Judge_Total')
fig.update_layout(title="Score moyen LLM-as-judge par modèle", xaxis_title="Modèle", yaxis_title="Score moyen", width=800, height=500)
fig.show()

In [19]:
avg_scores = (llm_human_score.groupby("Model")[["Human_Correct"]].mean().round(4).reset_index())

fig = px.bar(avg_scores, x='Model', y='Human_Correct')
fig.update_layout(title="Score moyen humain par modèle", xaxis_title="Modèle", yaxis_title="Score moyen", width=800, height=500)
fig.show()

## 4. Additional Visualizations

In [22]:
import plotly.express as px

# Pivot the data to get mean Judge_Total per Model and Subject
pivot_table = llm_human_score.pivot_table(values='Judge_Total', index='Model', columns='Subject', aggfunc='mean')

# Create the heatmap with Plotly
fig = px.imshow(pivot_table, 
                text_auto='.2f', 
                aspect="auto", 
                title='Heatmap of Mean LLM Judge Scores per Subject and Model',
                labels=dict(x="Subject", y="Model", color="Judge_Total"))
fig.show()

In [23]:
# Pivot the data to get mean Judge_Total per Model and Subject
pivot_table = llm_human_score.pivot_table(values='Human_Correct', index='Model', columns='Subject', aggfunc='mean')

# Create the heatmap with Plotly
fig = px.imshow(pivot_table, 
                text_auto='.2f', 
                aspect="auto", 
                title='Heatmap of Mean LLM Judge Scores per Subject and Model',
                labels=dict(x="Subject", y="Model", color="Human_Correct"))
fig.show()